In [ ]:
import numpy as np
import geopandas as gpd
import pandas as pd
from shapely.geometry import LineString
from shapely.ops import substring
from pysheds.grid import Grid
import rasterio
from rasterio.features import rasterize
from rasterio.windows import from_bounds
from rasterio.transform import rowcol
from scipy.ndimage import gaussian_filter, binary_erosion
import tempfile
import os
import math
import gc

# --- Adjustable parameters ---
SIGMA_VALUES_TO_TEST = [2.0]  # Extended for flat terrain
MINIMUM_PATH_LENGTH_FT = 100.0  # Minimum path length in feet
pit_min_depth = 0.05
depression_epsilon = 1e-6  # Finer depression filling for flats
flats_max_iter = 800000 # Increased for better flat resolution
BUFFER_FT = 75.0  # Buffer in feet for 2.5 ft DEM
BURN_DEPTH_FT = 1.5  # Burn depth in feet for stream burning

# --- Input files ---
basin_shp = r'C:\TC_Calculator_Work\Basin_BW.shp'
dem_tif = r'C:\TC_Calculator_Work\BW_SURF.tif'
output_shp = r'C:\TC_Calculator_Work\TC_Basin_BW_SIGMA2.shp'
stream_shp1 = r'C:\TC_Calculator_Work\NHD_H_03080102_HU8_Shape\Shape\NHDFlowline.shp'
stream_shp2 = r'C:\TC_Calculator_Work\NHD_H_03080101_HU8_Shape\Shape\NHDFlowline.shp'

# Load original DEM to get CRS
with rasterio.open(dem_tif) as src:
    dem_crs = src.crs

# Load basin and stream shapefiles and reproject if necessary
basins_gdf = gpd.read_file(basin_shp)
if dem_crs is not None and basins_gdf.crs != dem_crs:
    basins_gdf = basins_gdf.to_crs(dem_crs)
streams_gdf1 = gpd.read_file(stream_shp1)
streams_gdf2 = gpd.read_file(stream_shp2)
streams_gdf = pd.concat([streams_gdf1, streams_gdf2], ignore_index=True)  # Use pandas.concat
if dem_crs is not None and streams_gdf.crs != dem_crs:
    streams_gdf = streams_gdf.to_crs(dem_crs)

# Get total bounds with buffer and read the main DEM array once
total_bounds = basins_gdf.total_bounds  # [minx, miny, maxx, maxy]
buffered_bounds = (total_bounds[0] - BUFFER_FT, total_bounds[1] - BUFFER_FT, total_bounds[2] + BUFFER_FT, total_bounds[3] + BUFFER_FT)
with rasterio.open(dem_tif) as src:
    window = from_bounds(*buffered_bounds, transform=src.transform).round_offsets(op=math.floor).round_lengths(op=math.ceil)
    if window.width <= 0 or window.height <= 0:
        raise ValueError("Clipped window has zero size.")
   
    original_clipped_array = src.read(1, window=window).astype('float32')
    if original_clipped_array.size == 0:
        raise ValueError("Clipped array is empty.")
       
    meta = src.profile.copy()
    meta.update({
        'height': original_clipped_array.shape[0],
        'width': original_clipped_array.shape[1],
        'transform': rasterio.windows.transform(window, src.transform),
        'dtype': 'float32'
    })

# --- Helper functions ---
def custom_fill_pits(grid, dem, min_depth):
    pits = grid.detect_pits(dem)
    if not np.any(pits): return dem
    if not np.any(~pits): return grid.fill_pits(dem)
    min_non_pit = np.min(dem[~pits])
    pits_rows, pits_cols = np.where(pits)
    pit_depths = dem[pits_rows, pits_cols] - min_non_pit
    deep_pits = pit_depths >= min_depth
    shallow_rows, shallow_cols = pits_rows[~deep_pits], pits_cols[~deep_pits]
    filled = grid.fill_pits(dem)
    filled[shallow_rows, shallow_cols] = dem[shallow_rows, shallow_cols]
    return filled

def idx_to_xy(grid, row, col):
    return grid.affine * (col, row)

def calculate_slope(dem, path_coords, grid):
    # Convert x, y coordinates to row, col indices using the grid's affine transform
    rows, cols = rowcol(grid.affine, [coord[0] for coord in path_coords], [coord[1] for coord in path_coords])
    elevations = [dem[row, col] for row, col in zip(rows, cols) if 0 <= row < dem.shape[0] and 0 <= col < dem.shape[1]]
    if len(elevations) < 2:
        return 0.0
    elevation_drop = elevations[0] - elevations[-1]
    path_length = LineString(path_coords).length
    return elevation_drop / path_length if path_length > 0 else 0.0

def burn_streams(dem, streams_gdf, transform, shape, burn_depth):
    """Manually burn streams into DEM by lowering elevations."""
    dem_burned = dem.copy()
    for geom in streams_gdf.geometry:
        if geom.geom_type == 'LineString':  # Use geom_type instead of type
            x_coords, y_coords = geom.coords.xy
            rows, cols = rowcol(transform, x_coords, y_coords)
            for r, c in zip(rows, cols):
                if 0 <= r < shape[0] and 0 <= c < shape[1]:
                    dem_burned[r, c] = max(dem_burned[r, c] - burn_depth, -9999)  # Avoid extreme negatives
    return dem_burned

deltas = {1:(0, 1), 2:(1, 1), 4:(1, 0), 8:(1, -1), 16:(0, -1), 32:(-1, -1), 64:(-1, 0), 128:(-1, 1)}

def trace_path(fdir, start_row, start_col, outlet_row, outlet_col, deltas, mask):
    path = []
    r, c = start_row, start_col
    for _ in range(fdir.size):
        path.append((r, c))
        if (r, c) == (outlet_row, outlet_col) or not mask[r, c]: break
        dir_code = fdir[r, c]
        if dir_code not in deltas: break
        dr, dc = deltas[dir_code]
        r, c = r + dr, c + dc
        if not (0 <= r < fdir.shape[0] and 0 <= c < fdir.shape[1]): break
    return path

# --- Main Iterative Processing Loop ---
# Initialize dictionary to store candidates for each basin across all sigmas
basin_ids = basins_gdf['id'].tolist() if 'id' in basins_gdf.columns else list(range(len(basins_gdf)))
candidates = {basin_id: [] for basin_id in basin_ids}

for sigma in SIGMA_VALUES_TO_TEST:
    print(f"\n{'='*60}\n--- RUNNING ANALYSIS WITH SIGMA = {sigma} ---\n{'='*60}")
  
    # Burn streams into the original clipped array before smoothing
    burned_array = burn_streams(original_clipped_array.copy(), streams_gdf, meta['transform'], original_clipped_array.shape, BURN_DEPTH_FT)
    clipped_array = gaussian_filter(burned_array, sigma=sigma, mode='nearest')
  
    with tempfile.NamedTemporaryFile(suffix='.tif', delete=False) as tmpfile:
        temp_path = tmpfile.name
    with rasterio.open(temp_path, 'w', **meta) as dst:
        dst.write(clipped_array, 1)
  
    grid = Grid.from_raster(temp_path)
    dem = grid.read_raster(temp_path)
    
    pit_filled_dem = custom_fill_pits(grid, dem, pit_min_depth)
    depress_filled_dem = grid.fill_depressions(pit_filled_dem, epsilon=depression_epsilon)
    inflated_dem = grid.resolve_flats(depress_filled_dem, max_iter=flats_max_iter)
    fdir = grid.flowdir(inflated_dem)
    acc = grid.accumulation(fdir)
    gc.collect()
  
    print(f"--- Processing basins for sigma = {sigma} ---")
    for idx, row in basins_gdf.iterrows():
        basin_geom = row.geometry
        basin_id = row.get('id', idx)  # Use 'id' column if present, else index
        mask = rasterize([basin_geom], out_shape=acc.shape, transform=grid.affine, fill=0, default_value=1, dtype='uint8')
        eroded_mask = binary_erosion(mask)
        boundary_mask = mask - eroded_mask
        acc_on_boundary = acc * boundary_mask
      
        if np.all(acc_on_boundary == 0):
            print(f" -> Basin {basin_id} skipped: No valid outlet found.")
            continue
          
        outlet_idx = np.unravel_index(np.argmax(acc_on_boundary), acc.shape)
        outlet_row, outlet_col = outlet_idx
      
        dist = grid.distance_to_outlet(x=outlet_col, y=outlet_row, fdir=fdir, xytype='index')
        dist[np.isinf(dist)] = np.nan
        dist_basin = dist * mask
        dist_basin[np.isinf(dist_basin) | (mask == 0)] = np.nan
        if np.all(np.isnan(dist_basin)):
            print(f" -> Basin {basin_id} skipped: No valid flow path found.")
            continue
       
        start_idx = np.unravel_index(np.nanargmax(dist_basin), acc.shape)
        start_row, start_col = start_idx
        path_indices = trace_path(fdir, start_row, start_col, outlet_row, outlet_col, deltas, mask)
        path_coords = [idx_to_xy(grid, r, c) for r, c in path_indices]
      
        if len(path_coords) < 2:
            print(f" -> Basin {basin_id} skipped: Path too short.")
            continue
          
        line = LineString(path_coords)
        total_length = line.length
      
        if total_length >= MINIMUM_PATH_LENGTH_FT:
            print(f" -> SUCCESS for Basin {basin_id} with sigma = {sigma}. Length = {total_length:.2f} ft")
            slope = calculate_slope(dem, path_coords, grid)
            candidates[basin_id].append({
                'geometry': line,
                'length': total_length,
                'slope': slope,
                'sigma_used': sigma
            })
        else:
            print(f" -> Basin {basin_id} has short path ({total_length:.2f} ft). Skipping for this sigma.")

    os.remove(temp_path)

# --- Post-Processing: Select Best Line (Longest Path) ---
final_lines = {}
for basin_id in candidates:
    basin_candidates = candidates[basin_id]
    if basin_candidates:
        best_line = max(basin_candidates, key=lambda x: x['length'])  # Select longest path
        total_length = best_line['length']
        line = best_line['geometry']
        slope = best_line['slope']
        sigma = best_line['sigma_used']
        sheet_dist = 100.0
        if total_length <= sheet_dist:
            final_lines[basin_id] = [{
                'basin_id': basin_id,
                'type': 'sheet_flow',
                'geometry': line,
                'length': total_length,
                'sigma_used': sigma,
                'slope': slope
            }]
        else:
            sheet_line = substring(line, 0, sheet_dist)
            conc_line = substring(line, sheet_dist, total_length)
            final_lines[basin_id] = [
                {
                    'basin_id': basin_id,
                    'type': 'sheet_flow',
                    'geometry': sheet_line,
                    'length': sheet_line.length,
                    'sigma_used': sigma,
                    'slope': slope
                },
                {
                    'basin_id': basin_id,
                    'type': 'shallow_conc',
                    'geometry': conc_line,
                    'length': conc_line.length,
                    'sigma_used': sigma,
                    'slope': slope
                }
            ]
    else:
        print(f" -> Basin {basin_id} has no valid path across all sigma values. Assigning default.")
        # Assign default for basins with no valid path
        final_lines[basin_id] = [{
            'basin_id': basin_id,
            'type': 'sheet_flow',
            'geometry': LineString(),
            'length': MINIMUM_PATH_LENGTH_FT,
            'sigma_used': 'default',
            'slope': 0.0
        }]

# --- Final Output ---
output_list = [item for sublist in final_lines.values() for item in sublist]
if output_list:
    output_gdf = gpd.GeoDataFrame(output_list, crs=basins_gdf.crs)
    output_gdf.to_file(output_shp)
    print(f"\nOutput shapefile saved to {output_shp}")
    print(f"Successfully generated paths for {len(final_lines)} out of {len(basins_gdf)} basins.")
else:
    print("\nNo significant flow lines were generated that met the minimum length requirement.")
gc.collect()